# THOTH – Computer Vision, Week 1: Age & Emotion Model Comparison

**What this notebook does**
1. Detects every face in your test photos (multi-face) with one shared face detector (SCRFD from InsightFace),
   so all age/emotion models see exactly the same face crops — a fair comparison.
2. Runs **several pretrained open-source age models** and **several emotion models** on those crops (no training).
3. Maps outputs to our classes:
   - Age group: **Child (0–12), Teen (13–19), Adult (20–59), Senior (60+)**
   - Emotion: **Happy, Sad, Neutral, Bored, Excited**
4. Measures speed (ms per face) and saves comparison tables + labelled images for the report.
5. (Optional) If you fill in the true labels for each face, it computes **accuracy per model** and **per group**
   (hijab, dark skin, child, far away, …).

**How to run (Colab)**
- `Runtime → Change runtime type → T4 GPU`
- Run the cells top to bottom. After the install cell, if Colab asks to restart, restart and continue from cell 2.
- Everything is saved in `thoth_cv_week1/` and zipped at the end for download.

## 1. Install
Note: `hsemotion-onnx` pulls the CPU-only `onnxruntime`, which hides the GPU version (that's why the first
test notebook ran on CPU). We install it without dependencies and keep only `onnxruntime-gpu`.

In [ ]:
!pip install -q insightface deepface "transformers==4.51.0" "accelerate==1.8.1" opencv-python-headless pandas matplotlib
!pip install -q --no-deps hsemotion-onnx
!pip install -q --no-deps git+https://github.com/WildChlamydia/MiVOLO.git
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu

## 2. Imports, device check, settings

In [ ]:
import os, time, json, glob, math, shutil, warnings, traceback
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    matplotlib.use("Agg")

import onnxruntime as ort
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_CUDA = False

ORT_PROVIDERS = (["CUDAExecutionProvider", "CPUExecutionProvider"]
                 if "CUDAExecutionProvider" in ort.get_available_providers() else ["CPUExecutionProvider"])
print("onnxruntime providers:", ort.get_available_providers())
print("PyTorch CUDA available:", HAS_CUDA, "|", torch.cuda.get_device_name(0) if HAS_CUDA else "CPU only")
DEVICE_NAME = torch.cuda.get_device_name(0) if HAS_CUDA else "CPU"

# ---------- Settings ----------
IMG_DIR = Path("test_images")          # put your test photos here (or upload in the next cell)
OUT_DIR = Path("thoth_cv_week1")       # all results go here
IMG_DIR.mkdir(exist_ok=True); OUT_DIR.mkdir(exist_ok=True)

DET_SIZE = (640, 640)      # detector input size; bigger finds smaller/farther faces but is slower
DET_THRESHOLD = 0.5        # minimum face confidence
CROP_MARGIN = 0.15         # extra border around each face (15%), helps age/emotion models

AGE_GROUPS = [("Child", 0, 12), ("Teen", 13, 19), ("Adult", 20, 59), ("Senior", 60, 200)]
EMOTIONS_5 = ["Happy", "Sad", "Neutral", "Bored", "Excited"]

# Initial thresholds for the arousal-based rules (tuned later on our own clips — week 4)
EXCITED_AROUSAL = 0.30     # happy/surprised AND arousal above this  -> Excited
BORED_AROUSAL = -0.20      # neutral/contempt AND arousal below this -> Bored

## 3. Test images
Upload 20–30 photos: groups, kids, seniors, hijab, different skin tones, glasses, museum scenes, far/near,
dim light. **Use photos you have permission to use.** Name files descriptively if you can,
e.g. `group_hijab_dim.jpg`, `kids_museum.jpg` — the name appears in all tables.

In [ ]:
if IN_COLAB:
    uploaded = colab_files.upload()
    for name, data in uploaded.items():
        (IMG_DIR / name).write_bytes(data)

image_paths = sorted([p for p in IMG_DIR.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}])
if not image_paths:  # fallback so the notebook still runs: InsightFace's sample group photo
    import insightface.data as ins_data
    cv2.imwrite(str(IMG_DIR / "sample_group.jpg"), ins_data.get_image("t1"))
    image_paths = [IMG_DIR / "sample_group.jpg"]
print(f"{len(image_paths)} test images:", [p.name for p in image_paths])

## 4. Face detection (shared by all models)
SCRFD-10G from InsightFace's `buffalo_l` pack. We load **only** the detector
(`allowed_modules=['detection']`) instead of all 5 models in the pack — faster and less memory.
Detector comparison (SCRFD vs YuNet vs YOLO-face, …) is next week's task.

In [ ]:
from insightface.app import FaceAnalysis

detector = FaceAnalysis(name="buffalo_l", allowed_modules=["detection"], providers=ORT_PROVIDERS)
detector.prepare(ctx_id=0 if "CUDAExecutionProvider" in ORT_PROVIDERS else -1,
                 det_size=DET_SIZE, det_thresh=DET_THRESHOLD)

def crop_with_margin(img, bbox, margin=CROP_MARGIN):
    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox
    bw, bh = x2 - x1, y2 - y1
    x1 = int(max(0, x1 - margin * bw)); y1 = int(max(0, y1 - margin * bh))
    x2 = int(min(w, x2 + margin * bw)); y2 = int(min(h, y2 + margin * bh))
    return img[y1:y2, x1:x2]

images, faces = {}, []          # faces: one dict per detected face
det_rows = []
_ = detector.get(cv2.imread(str(image_paths[0])))  # warm-up
for p in image_paths:
    img = cv2.imread(str(p))
    if img is None:
        print("Could not read", p.name); continue
    images[p.name] = img
    t0 = time.perf_counter()
    dets = detector.get(img)
    det_ms = (time.perf_counter() - t0) * 1000
    dets = sorted(dets, key=lambda f: f.bbox[0])  # left-to-right so face numbers are easy to follow
    for i, f in enumerate(dets):
        bbox = f.bbox.astype(int).tolist()
        faces.append({"face_id": f"{p.stem}_f{i+1}", "image": p.name, "bbox": bbox,
                      "det_score": round(float(f.det_score), 3),
                      "face_px": int(min(bbox[2]-bbox[0], bbox[3]-bbox[1])),
                      "crop_bgr": crop_with_margin(img, bbox)})
    det_rows.append({"image": p.name, "width": img.shape[1], "height": img.shape[0],
                     "faces_found": len(dets), "detect_ms": round(det_ms, 1)})

det_df = pd.DataFrame(det_rows)
det_df.to_csv(OUT_DIR / "detection_per_image.csv", index=False)
print(det_df.to_string(index=False))
print(f"\nTotal faces: {len(faces)}")

### Face gallery (use it to fill in the true labels later)

In [ ]:
def save_gallery(faces, path, cols=8, size=128):
    if not faces: return
    rows = math.ceil(len(faces) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.8, rows * 2.1))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, f in zip(axes, faces):
        ax.imshow(cv2.cvtColor(cv2.resize(f["crop_bgr"], (size, size)), cv2.COLOR_BGR2RGB))
        ax.set_title(f["face_id"], fontsize=7)
    plt.tight_layout(); plt.savefig(path, dpi=120); plt.show() if IN_COLAB else plt.close()

save_gallery(faces, OUT_DIR / "face_gallery.png")

## 5. Label helpers (mapping every model to our classes)

In [ ]:
import re

def age_to_group(age):
    for name, lo, hi in AGE_GROUPS:
        if age <= hi: return name
    return "Senior"

def bucket_to_age(label):
    """'20-29' -> 24.5, 'more than 70' -> 75, '3-9' -> 6"""
    nums = [int(n) for n in re.findall(r"\d+", label)]
    if not nums: return np.nan
    if len(nums) == 1: return nums[0] + 5
    return (nums[0] + nums[1]) / 2

_EMO_ALIASES = {"angry": "anger", "anger": "anger", "disgust": "disgust", "fear": "fear",
                "happy": "happy", "happiness": "happy", "sad": "sad", "sadness": "sad",
                "surprise": "surprise", "neutral": "neutral", "contempt": "contempt"}
NEGATIVE = {"anger", "disgust", "fear", "sad"}

def norm_emotion(label):
    return _EMO_ALIASES.get(label.strip().lower(), label.strip().lower())

def map_to_5(probs, arousal=None):
    """probs: dict raw_emotion -> probability. arousal: float in [-1,1] or None.
    Returns one of Happy / Sad / Neutral / Bored / Excited."""
    top = max(probs, key=probs.get)
    if arousal is not None:
        if top in ("happy", "surprise") and arousal >= EXCITED_AROUSAL: return "Excited"
        if top == "happy": return "Happy"
        if top == "surprise": return "Excited"
        if top in ("neutral", "contempt") and arousal <= BORED_AROUSAL: return "Bored"
        if top in ("neutral", "contempt"): return "Neutral"
        if top in NEGATIVE: return "Sad"
    else:  # model has no arousal output -> it can never say Bored
        if top == "happy": return "Happy"
        if top == "surprise": return "Excited"
        if top in ("neutral", "contempt"): return "Neutral"
        if top in NEGATIVE: return "Sad"
    return "Neutral"

def bgr_to_rgb(x): return cv2.cvtColor(x, cv2.COLOR_BGR2RGB)

## 6. Candidate models
Each model is loaded inside `try/except`: if one fails to install/download, it is marked **FAILED** in the
status table (that is itself a finding for the report) and the rest continue.

Every predictor takes a list of **BGR** face crops (OpenCV format) and converts to what the model expects.
(The first test notebook passed BGR to HSEmotion, which expects RGB — fixed here.)

In [ ]:
AGE_MODELS, EMO_MODELS, STATUS = {}, {}, []

def register(task, name, loader, info):
    t0 = time.perf_counter()
    try:
        fn = loader()
        STATUS.append({"task": task, "model": name, "status": "loaded",
                       "load_s": round(time.perf_counter() - t0, 1), **info, "error": ""})
        (AGE_MODELS if task == "age" else EMO_MODELS)[name] = fn
        print(f"[OK]     {task:7s} {name}")
    except Exception as e:
        STATUS.append({"task": task, "model": name, "status": "FAILED",
                       "load_s": round(time.perf_counter() - t0, 1), **info, "error": f"{type(e).__name__}: {e}"[:300]})
        print(f"[FAILED] {task:7s} {name}: {type(e).__name__}: {str(e)[:200]}")

# ---------------- AGE MODELS ----------------
def load_insightface_age():
    from insightface.model_zoo import model_zoo
    from insightface.app.common import Face
    path = os.path.expanduser("~/.insightface/models/buffalo_l/genderage.onnx")
    m = model_zoo.get_model(path, providers=ORT_PROVIDERS)
    m.prepare(ctx_id=0)
    def predict(face_list):
        # genderage aligns using the bbox, so we run it on the full image with the original bbox
        out = []
        for f in face_list:
            fobj = Face(bbox=np.array(f["bbox"], dtype=np.float32))
            m.get(images[f["image"]], fobj)
            out.append({"age": float(fobj.age), "raw": str(int(fobj.age)), "conf": np.nan})
        return out
    return predict

def load_mivolo():
    from transformers import AutoModelForImageClassification, AutoImageProcessor
    dtype = torch.float16 if HAS_CUDA else torch.float32
    model = AutoModelForImageClassification.from_pretrained("iitolstykh/mivolo_v2", trust_remote_code=True,
                                                            torch_dtype=dtype)
    model = model.to("cuda" if HAS_CUDA else "cpu").eval()
    proc = AutoImageProcessor.from_pretrained("iitolstykh/mivolo_v2", trust_remote_code=True)
    def predict(face_list):
        crops = [f["crop_bgr"] for f in face_list]          # MiVOLO's processor expects BGR
        x = proc(images=crops)["pixel_values"].to(dtype=model.dtype, device=model.device)
        with torch.no_grad():
            out = model(faces_input=x, body_input=None)
        ages = out.age_output.float().cpu().numpy().reshape(-1)
        return [{"age": float(a), "raw": f"{a:.0f}", "conf": np.nan} for a in ages]
    return predict

def load_vit_age():
    from transformers import pipeline
    from PIL import Image
    pipe = pipeline("image-classification", model="nateraw/vit-age-classifier", device=0 if HAS_CUDA else -1)
    def predict(face_list):
        ims = [Image.fromarray(bgr_to_rgb(f["crop_bgr"])) for f in face_list]
        res = pipe(ims, top_k=1)
        res = [r[0] if isinstance(r, list) else r for r in res]
        return [{"age": bucket_to_age(r["label"]), "raw": r["label"], "conf": float(r["score"])} for r in res]
    return predict

def load_deepface_age():
    from deepface import DeepFace
    def predict(face_list):
        out = []
        for f in face_list:
            r = DeepFace.analyze(f["crop_bgr"], actions=["age"], detector_backend="skip",
                                 enforce_detection=False, silent=True)
            r = r[0] if isinstance(r, list) else r
            out.append({"age": float(r["age"]), "raw": str(r["age"]), "conf": np.nan})
        return out
    predict([faces[0]]) if faces else None  # triggers weight download inside load time
    return predict

# ---------------- EMOTION MODELS ----------------
def load_hsemotion(model_name):
    def loader():
        from hsemotion_onnx.facial_emotions import get_model_path
        sess = ort.InferenceSession(get_model_path(model_name), providers=ORT_PROVIDERS)
        size = 224 if "_b0_" in model_name else 260
        classes = (["anger", "disgust", "fear", "happy", "neutral", "sad", "surprise"] if "_7" in model_name else
                   ["anger", "contempt", "disgust", "fear", "happy", "neutral", "sad", "surprise"])
        is_va = "_mtl" in model_name
        mean, std = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])
        def predict(face_list):
            x = np.stack([((cv2.resize(bgr_to_rgb(f["crop_bgr"]), (size, size)) / 255.0 - mean) / std)
                          .transpose(2, 0, 1) for f in face_list]).astype(np.float32)
            scores = sess.run(None, {"input": x})[0]
            out = []
            for s in scores:
                logits = s[:-2] if is_va else s
                p = np.exp(logits - logits.max()); p /= p.sum()
                probs = dict(zip(classes, p.tolist()))
                v, a = (float(s[-2]), float(s[-1])) if is_va else (None, None)
                top = max(probs, key=probs.get)
                out.append({"raw": top, "conf": probs[top], "valence": v, "arousal": a,
                            "class5": map_to_5(probs, a)})
            return out
        return predict
    return loader

def load_hf_emotion(repo):
    def loader():
        from transformers import pipeline
        from PIL import Image
        pipe = pipeline("image-classification", model=repo, device=0 if HAS_CUDA else -1)
        def predict(face_list):
            ims = [Image.fromarray(bgr_to_rgb(f["crop_bgr"])) for f in face_list]
            res = pipe(ims, top_k=None)
            out = []
            for r in res:
                probs = {norm_emotion(d["label"]): float(d["score"]) for d in r}
                top = max(probs, key=probs.get)
                out.append({"raw": top, "conf": probs[top], "valence": None, "arousal": None,
                            "class5": map_to_5(probs)})
            return out
        return predict
    return loader

def load_deepface_emotion():
    from deepface import DeepFace
    def predict(face_list):
        out = []
        for f in face_list:
            r = DeepFace.analyze(f["crop_bgr"], actions=["emotion"], detector_backend="skip",
                                 enforce_detection=False, silent=True)
            r = r[0] if isinstance(r, list) else r
            probs = {norm_emotion(k): float(v) / 100 for k, v in r["emotion"].items()}
            top = max(probs, key=probs.get)
            out.append({"raw": top, "conf": probs[top], "valence": None, "arousal": None,
                        "class5": map_to_5(probs)})
        return out
    predict([faces[0]]) if faces else None
    return predict

# ---------------- Register everything ----------------
register("age", "InsightFace-genderage", load_insightface_age,
         {"source": "deepinsight/insightface (buffalo_l)", "trained_on": "CelebA-style data (not documented in detail)", "output": "exact age"})
register("age", "MiVOLO-v2", load_mivolo,
         {"source": "iitolstykh/mivolo_v2 (HF)", "trained_on": "LAGENDA + IMDB-clean + others", "output": "exact age"})
register("age", "ViT-age (FairFace)", load_vit_age,
         {"source": "nateraw/vit-age-classifier (HF)", "trained_on": "FairFace (race-balanced)", "output": "9 age buckets"})
register("age", "DeepFace-age", load_deepface_age,
         {"source": "serengil/deepface", "trained_on": "IMDB-WIKI", "output": "exact age"})

register("emotion", "HSEmotion-B0-VA", load_hsemotion("enet_b0_8_va_mtl"),
         {"source": "HSE-asavchenko/hsemotion", "trained_on": "AffectNet", "output": "8 emotions + valence/arousal"})
register("emotion", "HSEmotion-B0-AFEW", load_hsemotion("enet_b0_8_best_afew"),
         {"source": "HSE-asavchenko/hsemotion (first test notebook)", "trained_on": "AffectNet (+AFEW tuned)", "output": "8 emotions"})
register("emotion", "HSEmotion-B2", load_hsemotion("enet_b2_8"),
         {"source": "HSE-asavchenko/hsemotion", "trained_on": "AffectNet", "output": "8 emotions"})
register("emotion", "ViT-FER (trpakov)", load_hf_emotion("trpakov/vit-face-expression"),
         {"source": "trpakov/vit-face-expression (HF)", "trained_on": "FER2013", "output": "7 emotions"})
register("emotion", "ViT-FER (dima806)", load_hf_emotion("dima806/facial_emotions_image_detection"),
         {"source": "dima806/facial_emotions_image_detection (HF)", "trained_on": "FER2013-style", "output": "7 emotions"})
register("emotion", "DeepFace-emotion", load_deepface_emotion,
         {"source": "serengil/deepface", "trained_on": "FER2013", "output": "7 emotions"})

status_df = pd.DataFrame(STATUS)
status_df.to_csv(OUT_DIR / "model_status.csv", index=False)
status_df[["task", "model", "status", "load_s", "trained_on", "output", "error"]]

## 7. Run every model on every face (with timing)

In [ ]:
pred_rows = []

def run_models(models, task):
    for name, predict in models.items():
        try:
            predict(faces[:1])  # warm-up (first call is always slower)
            total_ms = 0.0
            for img_name in images:
                flist = [f for f in faces if f["image"] == img_name]
                if not flist: continue
                t0 = time.perf_counter()
                outs = predict(flist)
                if HAS_CUDA: torch.cuda.synchronize()
                ms = (time.perf_counter() - t0) * 1000
                total_ms += ms
                for f, o in zip(flist, outs):
                    row = {"task": task, "model": name, "face_id": f["face_id"], "image": img_name,
                           "raw": o["raw"], "conf": o.get("conf"), "ms_per_face": ms / len(flist)}
                    if task == "age":
                        row["age"] = o["age"]; row["label"] = age_to_group(o["age"])
                    else:
                        row["valence"] = o.get("valence"); row["arousal"] = o.get("arousal"); row["label"] = o["class5"]
                    pred_rows.append(row)
            print(f"{task:7s} {name:22s} done  ({total_ms / max(len(faces), 1):.1f} ms/face)")
        except Exception as e:
            print(f"{task:7s} {name:22s} ERROR during inference: {e}")
            traceback.print_exc()

if faces:
    run_models(AGE_MODELS, "age")
    run_models(EMO_MODELS, "emotion")
pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(OUT_DIR / "all_predictions.csv", index=False)
print(len(pred_df), "predictions saved")

## 8. Comparison tables
- **Side-by-side**: each face (row) × each model (column).
- **Agreement with majority**: with no ground truth yet, how often a model agrees with what most models say.
  (A rough sanity check only — the majority can be wrong; real accuracy comes from section 10.)
- **Speed**: ms per face on this machine. FPS for N faces ≈ 1000 / (detect_ms + N × ms_per_face of each model used).

In [ ]:
def side_by_side(task):
    d = pred_df[pred_df.task == task]
    if d.empty: return pd.DataFrame()
    show = d.assign(cell=d.apply(lambda r: f"{r['label']} ({r['raw']})" if task == "age"
                                 else f"{r['label']} ({r['raw']} {r['conf']:.2f})", axis=1))
    return show.pivot(index="face_id", columns="model", values="cell")

age_table, emo_table = side_by_side("age"), side_by_side("emotion")
age_table.to_csv(OUT_DIR / "age_side_by_side.csv"); emo_table.to_csv(OUT_DIR / "emotion_side_by_side.csv")

def summary(task):
    d = pred_df[pred_df.task == task]
    if d.empty: return pd.DataFrame()
    majority = d.groupby("face_id")["label"].agg(lambda s: s.value_counts().index[0])
    d = d.assign(agrees=d.apply(lambda r: r["label"] == majority[r["face_id"]], axis=1))
    s = d.groupby("model").agg(faces=("face_id", "count"),
                               ms_per_face=("ms_per_face", "mean"),
                               mean_conf=("conf", "mean"),
                               agree_with_majority=("agrees", "mean"))
    s["faces_per_sec"] = 1000 / s["ms_per_face"]
    s["label_distribution"] = d.groupby("model")["label"].agg(lambda x: dict(x.value_counts()))
    return s.round(3).sort_values("ms_per_face")

age_summary, emo_summary = summary("age"), summary("emotion")
age_summary.to_csv(OUT_DIR / "age_summary.csv"); emo_summary.to_csv(OUT_DIR / "emotion_summary.csv")
print("Device:", DEVICE_NAME)
print("\nAGE MODELS\n", age_summary.to_string())
print("\nEMOTION MODELS\n", emo_summary.to_string())

In [ ]:
age_table

In [ ]:
emo_table

## 9. Labelled images for the report
One figure per test image: a panel per model, each face boxed and labelled. Saved to `thoth_cv_week1/visuals/`.

In [ ]:
VIS_DIR = OUT_DIR / "visuals"; VIS_DIR.mkdir(exist_ok=True)
COLORS = {"Child": (255, 170, 0), "Teen": (0, 200, 255), "Adult": (0, 200, 0), "Senior": (200, 0, 200),
          "Happy": (0, 200, 0), "Excited": (0, 200, 255), "Neutral": (200, 200, 200), "Bored": (255, 170, 0), "Sad": (0, 0, 255)}

def draw(img, rows):
    vis = img.copy()
    scale = max(0.6, min(1.6, img.shape[1] / 900))
    for _, r in rows.iterrows():
        f = next(x for x in faces if x["face_id"] == r["face_id"])
        x1, y1, x2, y2 = f["bbox"]
        c = COLORS.get(r["label"], (0, 255, 0))
        cv2.rectangle(vis, (x1, y1), (x2, y2), c, max(1, int(2 * scale)))
        txt = f"{r['face_id'].split('_')[-1]} {r['label']}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5 * scale, 1)
        ty = y1 - 4 if y1 - th - 6 > 0 else y2 + th + 4
        cv2.rectangle(vis, (x1, ty - th - 3), (x1 + tw + 4, ty + 3), c, -1)
        cv2.putText(vis, txt, (x1 + 2, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.5 * scale, (0, 0, 0), 1, cv2.LINE_AA)
    return vis

for img_name, img in images.items():
    for task in ("age", "emotion"):
        d = pred_df[(pred_df.task == task) & (pred_df.image == img_name)] if not pred_df.empty else pd.DataFrame()
        models = list(d["model"].unique()) if not d.empty else []
        if not models: continue
        cols = min(3, len(models)); rows_n = math.ceil(len(models) / cols)
        fig, axes = plt.subplots(rows_n, cols, figsize=(6 * cols, 4.5 * rows_n))
        axes = np.array(axes).reshape(-1)
        for ax in axes: ax.axis("off")
        for ax, m in zip(axes, models):
            ax.imshow(bgr_to_rgb(draw(img, d[d.model == m])))
            ax.set_title(m, fontsize=12)
        fig.suptitle(f"{task.upper()} – {img_name}", fontsize=14)
        plt.tight_layout()
        plt.savefig(VIS_DIR / f"{Path(img_name).stem}_{task}.png", dpi=110)
        plt.show() if IN_COLAB else plt.close()

## 10. (Optional) Accuracy with true labels
1. Run the next cell — it saves `labels_template.csv`.
2. Open it in Excel/Sheets. Using `face_gallery.png`, fill for each face:
   - `true_age_group`: Child / Teen / Adult / Senior (your best judgement if the real age is unknown)
   - `true_emotion`: Happy / Sad / Neutral / Bored / Excited
   - `tags`: anything describing the face, separated by `;` — e.g. `hijab;dark_skin;glasses;far;low_light;side_view`
3. Save it as `labels.csv` and upload it in the cell after. You get accuracy per model, overall and per tag.

In [ ]:
tmpl = pd.DataFrame([{"face_id": f["face_id"], "image": f["image"], "face_px": f["face_px"],
                      "true_age_group": "", "true_emotion": "", "tags": ""} for f in faces])
tmpl.to_csv(OUT_DIR / "labels_template.csv", index=False)
if IN_COLAB: colab_files.download(str(OUT_DIR / "labels_template.csv"))
print("Saved", OUT_DIR / "labels_template.csv")

In [ ]:
labels_path = Path("labels.csv")
if IN_COLAB and not labels_path.exists():
    up = colab_files.upload()
    if up: labels_path.write_bytes(list(up.values())[0])

if labels_path.exists():
    labels = pd.read_csv(labels_path).fillna("")
    acc_rows, tag_rows = [], []
    for task, col in (("age", "true_age_group"), ("emotion", "true_emotion")):
        d = pred_df[pred_df.task == task].merge(labels[["face_id", col, "tags"]], on="face_id")
        d = d[d[col].astype(str).str.strip() != ""]
        if d.empty: continue
        d["correct"] = d["label"].str.lower() == d[col].str.strip().str.lower()
        for m, g in d.groupby("model"):
            acc_rows.append({"task": task, "model": m, "faces_labelled": len(g), "accuracy": round(g.correct.mean(), 3)})
        dt = d.assign(tag=d["tags"].astype(str).str.split(";")).explode("tag")
        dt["tag"] = dt["tag"].str.strip()
        dt = dt[dt["tag"] != ""]
        for (m, t), g in dt.groupby(["model", "tag"]):
            tag_rows.append({"task": task, "model": m, "tag": t, "n": len(g), "accuracy": round(g.correct.mean(), 3)})
    acc_df = pd.DataFrame(acc_rows).sort_values(["task", "accuracy"], ascending=[True, False])
    acc_df.to_csv(OUT_DIR / "accuracy_overall.csv", index=False)
    print(acc_df.to_string(index=False))
    if tag_rows:
        tag_df = pd.DataFrame(tag_rows)
        tag_pivot = tag_df.pivot_table(index=["task", "model"], columns="tag", values="accuracy")
        tag_pivot.to_csv(OUT_DIR / "accuracy_per_tag.csv")
        print("\nAccuracy per tag (fairness check):\n", tag_pivot.round(2).to_string())
else:
    print("No labels.csv yet — skip this for now, the comparison above still works.")

## 11. Download everything

In [ ]:
info = {"device": DEVICE_NAME, "onnx_providers": ORT_PROVIDERS, "n_images": len(images), "n_faces": len(faces),
        "det_size": DET_SIZE, "crop_margin": CROP_MARGIN,
        "excited_arousal": EXCITED_AROUSAL, "bored_arousal": BORED_AROUSAL,
        "date": time.strftime("%Y-%m-%d %H:%M")}
(OUT_DIR / "run_info.json").write_text(json.dumps(info, indent=2))
shutil.make_archive("thoth_cv_week1", "zip", OUT_DIR)
if IN_COLAB: colab_files.download("thoth_cv_week1.zip")
print("Saved thoth_cv_week1.zip")